# Notebook 2: Fine-tune CHURRO and score it

This trains a **LoRA adapter** for **CHURRO-3B**, a handwriting model trained on 100,000
historical pages, on whole pages from all three collections at once, and scores the result on the
team's 55 dev pages with the competition's own `metric.py`, per collection.

The training is tailored to the problems Lokesh's dev-set audit found. Each problem has a fix, and
each fix is a setting you can turn off to measure what it's worth (Step 4 lists them).

**Use an `ml.g5.xlarge` (or `ml.g6.xlarge`) notebook instance.** On a T4 (`ml.g4dn`) it is two to
three times slower, and training in fp16 there can fail with a `nan` loss.
Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Install the libraries

Run this after every start of the instance, and before any other cell.

In [ ]:
%pip install -q -r requirements.txt

import importlib.metadata
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

# The model also needs torchvision in exactly the version and build made for this kernel's
# PyTorch (2.8.0 goes with 0.23.0, 2.9.1 with 0.24.1, ...). conda_pytorch kernels have it.
def torchvision_works():
    return subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True).returncode == 0

torch_version, _, build = importlib.metadata.version("torch").partition("+")
if not torchvision_works():
    _, minor, patch = (torch_version.split(".") + ["0"])[:3]
    wanted = f"torchvision==0.{int(minor) + 15}.{patch}"
    sources = ([["--index-url", f"https://download.pytorch.org/whl/{build}"]] if build else []) + [[]]
    for source in sources:
        print(f"Installing {wanted} for PyTorch {torch_version}{'+' + build if build else ''} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wanted, *source])
        if torchvision_works():
            break
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchvision"])
    else:
        raise SystemExit("Couldn't install a torchvision that works with this kernel's PyTorch. Switch the "
                         "kernel to conda_pytorch_p310 (Kernel → Change Kernel) and run this cell again.")
print(f"PyTorch {importlib.metadata.version('torch')}, torchvision {importlib.metadata.version('torchvision')}: ready.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here

# Your bucket name is saved here, outside the repository, so it's typed once and
# updating the code (git pull) never touches it.
import json
SETTINGS = os.path.join(WORK, "my_settings.json")
def saved_setting(name, default=""):
    try:
        return json.load(open(SETTINGS)).get(name, default)
    except (OSError, ValueError):
        return default
def save_setting(name, value):
    try:
        current = json.load(open(SETTINGS))
    except (OSError, ValueError):
        current = {}
    current[name] = value
    json.dump(current, open(SETTINGS, "w"), indent=2)
print("Working folder:", WORK)

## Step 3. Check the GPU, the disk and the data

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise SystemExit("This instance has no GPU. Stop it and change its type to ml.g5.xlarge (README.md, Part 1).")
gpu = torch.cuda.get_device_properties(0)
bf16 = torch.cuda.get_device_capability()[0] >= 8
print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, {'bf16' if bf16 else 'fp16 (no bf16 on this GPU)'}")
if not bf16:
    print("WARNING: this GPU (a T4?) trains in fp16, which can give a nan loss with this model, and is 2-3x slower.\n"
          "  An ml.g5.xlarge or ml.g6.xlarge instance is the safer choice (README.md, Part 1).")
# The model is about 7.5 GB, downloaded once into churro-work/hf_cache. Check there's room first.
import glob
free = shutil.disk_usage(WORK).free / 1e9
downloaded = sum(os.path.getsize(f) for f in glob.glob(os.path.join(WORK, "hf_cache", "hub", "models--stanford-oval--churro-3B",
                                                                     "blobs", "*")) if not f.endswith(".incomplete")) / 1e9
needed = max(0.0, 7.5 - downloaded) + 3   # what's left of the model, plus room for a run's checkpoints
print(f"Disk: {free:.1f} GB free in ~/SageMaker; the model needs {max(0.0, 7.5 - downloaded):.1f} GB more")
if free < needed:
    biggest = subprocess.run("du -sh ~/SageMaker/* ~/SageMaker/*/* 2>/dev/null | sort -h | tail -8",
                             shell=True, capture_output=True, text=True).stdout
    raise SystemExit(f"Not enough disk space: {free:.1f} GB free, about {needed:.0f} GB needed.\n"
                     "Either make the volume bigger (Stop the instance, Edit, Volume size 50 GB, Start; your files are kept), "
                     "or delete something you don't need, e.g. ~/SageMaker/badger-finetune/hf_cache (the Kurrent pipeline's "
                     "9 GB model copy, downloaded again if that pipeline runs). The biggest folders:\n" + biggest)
missing = [n for n in ("train_items.csv", "dev_pages.csv", "parts.csv") if not os.path.exists(os.path.join(DATA, n))]
if missing:
    raise SystemExit(f"Missing {', '.join(missing)}. Run Notebook 1 first.")
import pandas as pd
items = pd.read_csv(os.path.join(DATA, "train_items.csv"), keep_default_na=False)
print("Training images:", ", ".join(f"{c} {n}" for c, n in items.category.value_counts().items()))

## Step 4. Choose the run and the settings

| `RUN` | What it does | About how long on `ml.g5.xlarge` |
|---|---|---|
| `"smoke"` | 12 images, 1 epoch, 3 dev pages. Only proves everything works. | 10–15 min the first time (mostly downloading the 7.5 GB model, once) |
| `"plain"` | No training and none of the reading fixes below, on all 55 dev pages. (Survey spreads are still read one page at a time; that's set in Notebook 1.) | 15–30 min |
| `"baseline"` | No training, but with the reading fixes below. The difference from `"plain"` is what the fixes are worth on their own. | 15–30 min |
| `"full"` | Trains with all the settings below, then scores all 55 dev pages. | 1 to 1.5 hours |

On an `ml.g4dn.xlarge`, multiply by about 2.5.

**How each problem in the dev-set audit is tackled.** The numbers are the audit's own problem
numbers.

| Audit problem | What this does about it | Setting |
|---|---|---|
| 1. Survey spreads read in the wrong order, stopped early, or read twice | Spreads are cut at the fold and read one page at a time (Notebook 1). A reading that comes out empty, far too short, or stuck repeating itself is read again in two parts, top and bottom. | `retry` |
| 2. Kade dev postcards are small, faint pencil | Small images are enlarged to the same detail as the others. In training, Kade images are sometimes shrunk and faded, so the model also learns from pages like the postcards. | `upscale`, `augment` |
| 3. Kade postmark line isn't on the page | Never trained on. On small cards (68% of the small training cards have one) the part every postmark line shares, `Postmark , 19`, is added: on average it saves more than it costs. | `postmark_guess` |
| 4. Dialect spelling (*jestern*, *bei die factry*) | No spelling correction anywhere; labels are learned exactly as written. | (always) |
| 5. Stained, bleed-through Dominy pages; `#` in labels | `#` is never something the model learns to write. Training images sometimes get a faint mirror image of themselves, like ink from the back. An optional contrast stretch. | `augment`, `enhance` |
| 6. Ledger symbols the labels leave out | Check marks and `×` are removed from the output; curly quotes and long dashes become plain ones. | (always) |
| 7. Few pages, short pages, and every collection a third of the score | Dominy's 27 training pages are used 4 times per epoch, each time with different random changes. | `oversample` |
| 8. Survey vocabulary and numbers | Learned from 365 survey training images. | (training) |
| 9. Labels that are machine drafts | Scores on the expert-typed dev pages are shown separately. | (scoring) |

README.md explains every setting in plain words. **Change one at a time**, and write each run's
settings and scores in the team's results table. Each combination gets its own folder in `runs/`;
running it again continues from its last saved epoch, and deleting its folder starts it over.

In [ ]:
RUN = "smoke"   # "smoke", then "plain", "baseline" and "full"

settings = {
    # training
    "epochs": 2,                         # passes through all the training images
    "lr": 2e-4,                          # learning rate: how big each correction is
    "lora_r": 16,                        # adapter size
    "lora_alpha": 32,                    # adapter strength; keep at 2 x lora_r
    "grad_accum": 8,                     # images per correction
    "augment": "all",                    # random changes to training images: "all", "none", or e.g. "dominy_accounts"
    "oversample": "dominy_accounts:4",   # use a collection's images several times per epoch ("" = none)
    "train_vision": 0,                   # 1 = also train the image side of the model
    # reading (training and scoring alike)
    "max_pixels": 1280 * 28 * 28,        # image detail: about a megapixel
    "upscale": 1,                        # 1 = enlarge small images (the Kade postcards)
    "enhance": "",                       # contrast stretch for these collections, e.g. "dominy_accounts"
    "retry": 1,                          # 1 = read a part again if its reading looks wrong
    "postmark_guess": 1,                 # 1 = add "Postmark , 19" on small Kade cards
    # scoring
    "curve_pages": 3,                    # dev pages per collection scored after every epoch
}
reading = {k: settings[k] for k in ("max_pixels", "upscale", "enhance", "retry", "postmark_guess")}
if RUN == "smoke":
    settings.update(epochs=1, max_train_items=12, dev_pages=3, curve_pages=0)
elif RUN == "plain":
    settings = {"epochs": 0, "max_pixels": settings["max_pixels"], "upscale": 0, "enhance": "", "retry": 0,
                "postmark_guess": 0}
elif RUN == "baseline":
    settings = {"epochs": 0, **reading}

short = {"epochs": "ep", "lr": "lr", "lora_r": "r", "augment": "aug", "oversample": "over", "train_vision": "vis",
         "max_pixels": "px", "upscale": "up", "enhance": "enh", "retry": "re", "postmark_guess": "pm"}
def tag(v):
    return str(v).replace("_accounts", "").replace("_letters", "").replace("_notes", "").replace(",", "+").replace(":", "x") or "none"
run_name = RUN if RUN != "full" else "full-" + "-".join(f"{short[k]}{tag(settings[k])}" for k in short)
run_dir = os.path.join(RUNS, run_name)
os.makedirs(run_dir, exist_ok=True)
print("This run's folder:", run_dir)

## Step 5. Train and score

The progress appears below and is saved to `train.log` in the run's folder. If the browser
disconnects, it keeps going while the instance runs; open `train.log` to see where it is. To stop
it, use Kernel → Interrupt Kernel.

What to look for: `[dev before training]` and `[dev after epoch N]` (a few dev pages, for the
curve), `loss` going down, then `[dev after training]` with the full dev score per collection,
and how many pages needed a second reading.

In [ ]:
args = {**settings, "data_dir": DATA, "out_dir": run_dir}
flags = " ".join(f"--{k} '{v}'" for k, v in args.items())
log = os.path.join(run_dir, "train.log")
python = sys.executable
!{python} {SCRIPTS}/finetune_churro.py {flags} 2>&1 | tee -a {log}

## Step 6. The scores

CER is the share of characters wrong (lower is better). Kaggle's score is the **mean of the three
collections**, each counting for a third. Runs you've done of `"plain"` and `"baseline"` appear
next to this one, so you can see what the reading fixes and the training each add.

Then the audit's length check, per collection: a page whose reading is under 0.7 of its label's
length probably stopped early or skipped part of the page; over 1.1, it probably repeated
something or read a part twice. Fewer of both is better.

In [ ]:
import json

summary = json.load(open(os.path.join(run_dir, "summary.json")))
table = pd.DataFrame({"this run": summary["dev_cer"]})
for other in ("plain", "baseline"):
    path = os.path.join(RUNS, other, "summary.json")
    if os.path.exists(path) and run_name != other:
        table[other] = pd.Series(json.load(open(path))["dev_cer"])
print("Dev CER (lower is better):")
print(table.round(3).to_string())
if summary.get("dev_cer_expert_typed_pages"):
    print("\nOn the expert-typed dev pages only:", {k: round(v, 3) for k, v in summary["dev_cer_expert_typed_pages"].items()})
print("\nLength check (pages):")
print(pd.DataFrame(summary["dev_problems"]).T.to_string())
if "train_minutes" in summary:
    print(f"\nTraining took {summary['train_minutes']} min, {summary['train_images_per_epoch']} images per epoch.")

curve = json.load(open(os.path.join(run_dir, "curve.json")))
if curve["dev_cer_by_epoch"]:
    print("\nA few dev pages per collection after each epoch (0 = before training):")
    print(pd.DataFrame(curve["dev_cer_by_epoch"]).round(3).to_string(index=False))
if curve["loss"]:
    import matplotlib.pyplot as plt
    loss = pd.DataFrame(curve["loss"])
    plt.figure(figsize=(6, 3)); plt.plot(loss.step, loss.loss); plt.xlabel("step"); plt.ylabel("training loss")
    plt.tight_layout(); plt.show()

## Step 7. Look at one page

The page that scored worst first; type any dev page name to see another. Green is what the
model wrote that isn't in the label; red, struck through, is what it missed or got wrong.

In [ ]:
import difflib
import html
from IPython.display import HTML, display

def coloured_diff(reference, prediction):
    """Green: what the model wrote that isn't in the label. Red, struck through: what it missed or got wrong."""
    out = []
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(None, reference, prediction, autojunk=False).get_opcodes():
        ref, pred = html.escape(reference[a0:a1]), html.escape(prediction[b0:b1])
        if op == "equal":
            out.append(ref)
        if op in ("delete", "replace"):
            out.append(f'<span style="background:#f7c5c5;text-decoration:line-through">{ref}</span>')
        if op in ("insert", "replace"):
            out.append(f'<span style="background:#c8f7c5">{pred}</span>')
    return "".join(out).replace("\n", "<br>")

box = "font-family:monospace;line-height:1.6;border:1px solid #ccc;padding:8px"

scores = pd.read_csv(os.path.join(run_dir, "dev_scores.csv"), keep_default_na=False)
flagged = scores[scores.problem != ""].sort_values("cer", ascending=False)
if len(flagged):
    print("Pages the length check flagged:")
    print(flagged[["page_id", "cer", "length_ratio", "problem", "read_twice"]].round(3).to_string(index=False))
PAGE = scores.sort_values("cer").iloc[-1]["page_id"]   # or type a page name, e.g. "dominy_018_p002"
row = scores.set_index("page_id").loc[PAGE]
from PIL import Image
img = Image.open(os.path.join(DATA, "images", PAGE + ".jpg"))
img.thumbnail((700, 900))
display(img)
display(HTML(f"<h4>{PAGE}: CER {row['cer']:.3f}</h4><div style='{box}'>"
             f"{coloured_diff(row['reference'], row['prediction'])}</div>"))

## Step 8. Save the run to S3 (recommended)

The adapter is about 120 MB; the big checkpoint files are left out.

In [ ]:
MY_BUCKET = saved_setting("bucket")   # the bucket from Notebook 1; or type another, e.g. "my-bucket"

!aws s3 sync {run_dir} s3://{MY_BUCKET}/churro-work/runs/{run_name} --exclude "checkpoints/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/churro-work/runs/{run_name}")

**Next:** change one setting and run Steps 4 to 8 again, or go to **Notebook 3** to read the
test pages with your best run. First shut this notebook's kernel down (Kernel → Shut Down Kernel)
so the GPU is free. **When you stop for the day, stop the instance.**